# Isochrone CMD Plots
Generates color magnitude diagrams with MIST isochrone overlays for the four
clusters with multiple seismic detections: NGC 752, Theia 6046, Casado Alessi 1,
and Theia 844. Plots show seismic age range (solid) and literature age range (dashed).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import types
from isochrones import get_ichrone

STAR_FILE = '../../GitHub/Isochrones/final!_errors.csv'  # adjust path
df = pd.read_csv(STAR_FILE, low_memory=False)
outdir = '../../Downloads/revision_1'

cluster_col = next((c for c in df.columns if 'cluster' in c.lower()), None)

def pick(df_sub, col_patterns):
    cols = df_sub.columns; cols_lower = cols.str.lower()
    for col, cl in zip(cols, cols_lower):
        if all(p.lower() in cl for p in col_patterns): return col
    return None

mist = get_ichrone('mist', bands=['G', 'BP', 'RP'])
def patched_interp_value(self, pars, props):
    i0, i1, i2, i3, i4 = self.param_index_order
    pars = [pars[i0], pars[i1], pars[i2]]
    return self.model_grid.interp(pars, 'all')
mist.interp_value = types.MethodType(patched_interp_value, mist)

K_G = 2.74; K_BP = 3.37; K_RP = 2.04
print('Ready.')

In [ ]:
def plot_iso(ax, age, feh, EBV, DM, color, lw, alpha, ls, label, zorder=3):
    logage = np.log10(age * 1e9)
    iso = mist.isochrone(logage, feh)
    col_fit = (iso['BP_mag'] - iso['RP_mag']) + (K_BP - K_RP) * EBV
    mag_fit = iso['G_mag'] + DM + K_G * EBV
    ax.plot(col_fit, mag_fit, color=color, lw=lw, alpha=alpha, linestyle=ls, label=label, zorder=zorder)

clusters = {
    'NGC_752': {
        'match': 'NGC_752', 'label': 'NGC 752',
        'EBV': 0.03, 'DM': 8.25, 'feh': 0.08,
        'seis': (1.45, 1.71), 'lit': (1.00, 1.60),
        'xlim': (0.35, 1.45), 'ylim': (14, 9.5), 'fname': 'ngciso.png',
    },
    'Theia_6046': {
        'match': 'theia_6046', 'label': 'Theia 6046',
        'EBV': 0.10, 'DM': 9.6, 'feh': -0.06,
        'seis': (2.59, 6.69), 'lit': (2.50, 4.50),
        'xlim': (0.3, 2.0), 'ylim': (17, 10), 'fname': 'theiaiso.png',
    },
    'Casado_Alessi_1': {
        'match': 'Casado-Alessi_1', 'label': 'Casado Alessi 1',
        'EBV': 0.03, 'DM': 9.0, 'feh': 0.02,
        'seis': (0.88, 1.09), 'lit': (0.80, 1.50),
        'xlim': (0.2, 1.5), 'ylim': (16, 9.5), 'fname': 'casadoiso.png',
    },
    'Theia_844': {
        'match': 'Theia_844', 'label': 'Theia 844',
        'EBV': 0.08, 'DM': 9.0, 'feh': 0.0,
        'seis': (0.24, 0.28), 'lit': (0.10, 0.40),
        'xlim': (0.0, 1.2), 'ylim': (16, 9.5), 'fname': 'theia844iso.png',
    },
}

for cl_key, params in clusters.items():
    mask = df[cluster_col].astype(str).str.contains(params['match'], case=False, na=False)
    df_cl = df.loc[mask].copy()
    g_col = pick(df_cl, ['gmag']) or pick(df_cl, ['phot_g']) or pick(df_cl, ['g_mean_mag'])
    bp_col = pick(df_cl, ['bp', 'mag']); rp_col = pick(df_cl, ['rp', 'mag'])
    df_cl['BP_RP'] = df_cl[bp_col] - df_cl[rp_col]
    color_arr = df_cl['BP_RP'].to_numpy()
    gmag = pd.to_numeric(df_cl[g_col], errors='coerce').to_numpy()
    good = np.isfinite(color_arr) & np.isfinite(gmag)

    fig, ax = plt.subplots(figsize=(9, 8))
    ax.scatter(color_arr[good], gmag[good], s=15, color='gray', edgecolors='k', linewidth=0.3, alpha=0.5)
    slo, shi = params['seis']; llo, lhi = params['lit']
    plot_iso(ax, slo, params['feh'], params['EBV'], params['DM'], '#1b9e77', 2.5, 0.9, '-',
             f'Seismic ({slo:.2f}\u2013{shi:.2f} Gyr)')
    plot_iso(ax, shi, params['feh'], params['EBV'], params['DM'], '#1b9e77', 2.5, 0.9, '-', None)
    plot_iso(ax, llo, params['feh'], params['EBV'], params['DM'], '#d95f02', 2.5, 0.9, '--',
             f'Literature ({llo:.2f}\u2013{lhi:.2f} Gyr)')
    plot_iso(ax, lhi, params['feh'], params['EBV'], params['DM'], '#d95f02', 2.5, 0.9, '--', None)
    ax.invert_yaxis()
    ax.set_xlim(params['xlim']); ax.set_ylim(params['ylim'])
    ax.set_xlabel(r'$G_{\rm BP} - G_{\rm RP}$ (mag)', fontsize=13)
    ax.set_ylabel(r'$G$ (mag)', fontsize=13)
    ax.legend(fontsize=11, loc='lower left')
    ax.grid(alpha=0.2); ax.set_title(params['label'], fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{outdir}/{params["fname"]}', dpi=200, bbox_inches='tight')
    plt.show()
    print(f'Saved {params["fname"]}')